### Practice with GNNs

The purpose of this homework is for you to build intuition about graph neural networks. You will explore three different neural network learning approaches

- A graph autoencoder that uses one-hot encodings for the node feature vector
- A graph autoencoder that uses bag-of-words node features
- A graph convolutional neural network that uses node features and labeled training data

---

#### Imports and Utilities

**Imports:** Import all libraries that will be used in this notebook. Sort them into categories so that it's easy to see how they might be used.

In [ ]:
# GNN libraries
import torch
from torch_geometric.data import Data
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE

# Numpy and random
import random
import numpy as np

# Matplotlib utilities
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.axes import Axes

# NetworkX
import networkx as nx

# Useful utilities for understanding results
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Data and type management
from numpy.typing import NDArray
from typing import Hashable
from collections import defaultdict
import os
import pandas as pd


**Drawing Utilities:** Functions that we'll use to show results. We'll look at networks with labeled nodes so we'll need utilities for plotting networks with labeled nodes and with colored nodes. It will also be helpful to inspect clusters of the embeddings using a scatterplot.

In [ ]:
def plot_scatter(embeddings, labels, title="t-SNE of Node Embeddings", cmap=plt.cm.tab10):
    """Plot a 2D t-SNE scatterplot of node embeddings colored by labels."""
    z = TSNE(n_components=2, perplexity=5, random_state=42).fit_transform(embeddings)
    plt.figure(figsize=(8, 6))
    _ = plt.scatter(z[:, 0], z[:, 1], c=labels, cmap=cmap, s=60, edgecolors='k')
    plt.title(title)
    plt.axis('off')
    plt.show()


def plot_ground_truth_graph(G, node_labels, title="Graph with Node Labels", cmap=plt.cm.tab10):
    """Visualize a NetworkX graph with nodes colored by labels."""
    unique_labels_local = sorted(set(node_labels))
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8, 6))
    nx.draw(G, pos, node_color=node_labels, cmap=cmap, node_size=100, with_labels=False)
    plt.title(title)
    plt.axis('off')

    denom = max(len(unique_labels_local) - 1, 1)
    legend_handles = [
        Patch(color=plt.cm.tab10(i / denom), label=str(label))
        for i, label in enumerate(unique_labels_local)
    ]
    plt.legend(handles=legend_handles, title="Subjects", bbox_to_anchor=(1.05, 1), loc="upper left")


def plot_graph(G: nx.Graph,
               node_labels: list[int],
               pos: dict[Hashable, tuple[float, float]] | None = None,
               title: str = " ",
               ax: Axes | None = None) -> None:
    if pos is None:
        pos = nx.spring_layout(G, seed=42)
    if ax is None:
        plt.figure()
        ax = plt.gca()
    nx.draw(G,
            pos=pos,
            node_color=node_labels,
            cmap=plt.cm.tab10,
            node_size=100,
            ax=ax,
            with_labels=False)
    ax.set_title(title)


#### The Dataset

We'll construct a smaller dataset from the [Cora dataset](https://paperswithcode.com/dataset/cora). ChatGPT-4o helped with code to subsample this dataset to something more manageable.

**Import:** Import information about the nodes. The dataset assigns a unique paper ID to each paper. The dataset also includes a bag-of-words feature vector with 1433 words. Label the columns in the dataset with "feature_num". Label the last column with the class of the paper, which in the Cora dataset is the subject area into which the paper was categorized.

In [ ]:
# Load node data
DATASET_DIR = 'datasets'
if not os.path.exists(os.path.join(DATASET_DIR, 'cora.content')):
    DATASET_DIR = 'data'

column_names = ['paper_id'] + [f'feature_{i}' for i in range(1433)] + ['subject']
nodes: pd.DataFrame = pd.read_csv(
    os.path.join(DATASET_DIR, 'cora.content'),
    sep='\t',
    header=None,
    names=column_names
)
nodes.head()


Load the edges in the dataset, which is given by which papers cite which other papers. 

In [ ]:
# Load edge data
edges: pd.DataFrame = pd.read_csv(
    os.path.join(DATASET_DIR, 'cora.cites'),
    sep='\t',
    header=None,
    names=['source', 'target']
)
edges.head()


---

#### Build Networkx Graph

**Create Graph from DataFrame:** Create an undirected networkx graph. Nodes are indexed by the paper ID

In [ ]:
# Create a graph
G: nx.Graph = nx.from_pandas_edgelist(edges, 'source', 'target', create_using=nx.Graph())

# Add node attributes
for _, row in nodes.iterrows():
    G.nodes[row['paper_id']].update(row.to_dict())

# Show first 5 entries
num_nodes: int = 5
for node in G.nodes():
    print(f"node {node} is in class {G.nodes[node]['subject']} = {G.nodes[node]['feature_0']}")
    num_nodes -=1
    if num_nodes == 0:
        break


**Create a connected subgraph:** The full Cora database makes it a bit difficult to see results, so create a connected subgraph.

In [ ]:
# Code from ChatGPT in response to prompt of how to create smaller dataset
# with from Cora while ensuring that the resulting graph is connected
def snowball_sample(G_full: nx.Graph, start_node: str, target_size: int) -> nx.Graph:
    visited = set([start_node])
    frontier = set(G_full.neighbors(start_node))
    
    while len(visited) < target_size and frontier:
        next_node = random.choice(list(frontier))
        visited.add(next_node)
        frontier.update(G_full.neighbors(next_node))
        frontier -= visited  # remove already visited nodes
    
    return G_full.subgraph(visited).copy()

In [ ]:
# Ensure consistent results for everyone doing the assignment
random.seed(42)

# Start from a high-degree node
start_node = max(G.degree, key=lambda x: x[1])[0]

# Sample 250 connected nodes
G_sub = snowball_sample(G, start_node=start_node, target_size=250)

Visualize

In [ ]:
# Map class labels to integers
labels = nx.get_node_attributes(G_sub, 'subject')
unique_labels = sorted(set(labels.values()))
label_to_int = {label: i for i, label in enumerate(unique_labels)}

# Create a color list for the nodes
node_colors = [label_to_int[labels[node]] for node in G_sub.nodes()]

# Show
plot_ground_truth_graph(G_sub, node_colors, title="Cora Subgraph (250 Nodes) Colored by Class Label")


Notice how some classes don't have as many samples as others. Indeed, there are no nodes from the _Probabilistic Methods_ class.

In [ ]:
class_count: dict[str, int] = {subject: 0 for subject in set(nodes['subject'])}
for node in G_sub.nodes():
    class_count[G_sub.nodes[node]['subject']] += 1
print(class_count)

---

### Build PyTorch Data Structures

We will explore three types of models:
- Graph autoencoders that use one-hot encoding for each node to create a node embedding
- Graph autoencoders that use the node's feature vector to create a node embedding
- Graph convolutional neural networks that use the node's feature vector and some hand-labeled nodes to do semi-supervised learning.

Each model will be built using a class in the PyTorch Geometric library. The input to these models will be the data from the graph, and this data must be put into the format required by PyTorch. The dataformat is abbreviated _PyG_ for PyTorch Geometric. It needs to have edge information (since we are doing graph neural networks), node features, node labels, and information about training data (if using semi-supervised learning).

I used ChatGPT-4o to put the data into the correct format. Two formats are sufficient: one with one-hot encoding and one with node features plus training data.

**One-hot Encoding**

Build PyG data structure for data using one-hot encoding. 


In [ ]:
from torch_geometric.data import Data as PyGData

# Step 1: One-hot encode the 250 nodes in the subgraph
ordered_nodes = list(G_sub.nodes())
num_nodes = len(ordered_nodes)
x = torch.eye(num_nodes)  # Shape: (250, 250), each row is a one-hot vector

# Step 2: Build node ID → index mapping
node_to_index = {node_id: i for i, node_id in enumerate(ordered_nodes)}

# Step 3: Remap edges using index mapping
edges = [
    [node_to_index[src], node_to_index[dst]]
    for src, dst in G_sub.edges()
]
edge_index = torch.tensor(edges, dtype=torch.long).T  # Shape: (2, num_edges)

# Step 4: Map class labels to integers
labels = nx.get_node_attributes(G_sub, 'subject')
unique_labels = sorted(set(labels.values()))
label_to_index = {label: i for i, label in enumerate(unique_labels)}
y = torch.tensor([label_to_index[labels[node]] for node in ordered_nodes], dtype=torch.long)

# Step 5: Create PyG Data object
data_onehot: PyGData = PyGData(x=x, edge_index=edge_index, y=y)

# Inspect
print(f"x shape: {data_onehot.x.shape}, edge_index shape: {data_onehot.edge_index.shape}, y shape: {data_onehot.y.shape}")

**Node Feature Vector**

Encode the feature of the input using the bag-of-words labels in the Cora dataset.

In [ ]:
# Step 1: Prepare node features and labels
X = nodes.iloc[:, 1:-1].values  # 1433 BoW features
y = nodes["subject"]
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Map paper_id to index
paper_id_to_idx = {paper_id: i for i, paper_id in enumerate(nodes["paper_id"])}

# Build edge index for subgraph
ordered_nodes = list(G_sub.nodes())
node_to_index = {node_id: i for i, node_id in enumerate(ordered_nodes)}
edge_list = [
    [node_to_index[src], node_to_index[dst]]
    for src, dst in G_sub.edges()
    if src in node_to_index and dst in node_to_index
]
edge_index = torch.tensor(edge_list, dtype=torch.long).T

# Build node feature matrix and label vector (aligned with ordered_nodes)
X_sub = torch.tensor([X[paper_id_to_idx[n]] for n in ordered_nodes], dtype=torch.float)
y_sub_raw = [y_encoded[paper_id_to_idx[n]] for n in ordered_nodes]

# Remap labels to contiguous IDs (0..num_classes-1) for cross-entropy.
unique_sub_labels = sorted(set(y_sub_raw))
label_remap = {old_label: new_label for new_label, old_label in enumerate(unique_sub_labels)}
y_sub = torch.tensor([label_remap[label] for label in y_sub_raw], dtype=torch.long)

data_cora: PyGData = PyGData(x=X_sub, edge_index=edge_index, y=y_sub)

In the GCN tutorial, I made a big deal about adding self-loops. That was to help you learn that graph convolutional neural networks work best if they can aggregate information from nearby nodes with each node's current signal.

We don't actually have to add self-loops explicitly when we use the `GCNConv` class because its default settings add self-loops if they are not already present.

**Node Labels and Masks**

We'll eventually use the node labels, so let's choose a subset of nodes in each class for labeled training data. To avoid leakage and to mirror standard semi-supervised practice in PyTorch Geometric, we'll create three masks:
- `train_mask`: nodes used for gradient updates
- `val_mask`: nodes used for model selection and tuning
- `test_mask`: nodes used once for final evaluation

We'll use approximately 10% training, 20% validation, and 70% test data, stratified by class.

In [ ]:
# Build train/val/test masks with class stratification
train_percent = 0.10
val_percent = 0.20
seed = 42

num_total = len(y_sub)
num_classes = int(y_sub.max().item()) + 1

rng = np.random.default_rng(seed)

# Group indices by class
class_indices = defaultdict(list)
for idx, label in enumerate(y_sub.tolist()):
    class_indices[label].append(idx)

train_indices = []
val_indices = []
test_indices = []

for _, indices in class_indices.items():
    indices = np.array(indices, dtype=int)
    rng.shuffle(indices)

    n = len(indices)
    n_train = max(1, int(train_percent * n))
    n_val = max(1, int(val_percent * n))

    # Ensure each class has at least one test example when possible.
    if n_train + n_val >= n:
        n_val = max(1, n - n_train - 1)

    train_cls = indices[:n_train]
    val_cls = indices[n_train:n_train + n_val]
    test_cls = indices[n_train + n_val:]

    train_indices.extend(train_cls.tolist())
    val_indices.extend(val_cls.tolist())
    test_indices.extend(test_cls.tolist())

train_mask = torch.zeros(num_total, dtype=torch.bool)
val_mask = torch.zeros(num_total, dtype=torch.bool)
test_mask = torch.zeros(num_total, dtype=torch.bool)

train_mask[train_indices] = True
val_mask[val_indices] = True
test_mask[test_indices] = True

data_cora.train_mask = train_mask
data_cora.val_mask = val_mask
data_cora.test_mask = test_mask

print(f"num classes: {num_classes}")
print(f"train nodes: {int(data_cora.train_mask.sum())}")
print(f"val nodes:   {int(data_cora.val_mask.sum())}")
print(f"test nodes:  {int(data_cora.test_mask.sum())}")


---

#### GNN Architecture

To make comparisons fair, we'll use essentially the same architecture for each GNN we build. The figure below was generated by ChatGPT-4o with prompts by me. It has obvious errors but illustrates the basic idea.

<img src="figures/Graph_Convolutional_Network_Architecture.png" alt="General architecture for each GNN" width = "800">

The architecture has two hidden layers, and the nonlinear "squashing" function between each layer will be an ReLU. Each GNN will use the same dimension for the hidden layers and for the output (except for the semi-supervised implementation). 

In [ ]:
HIDDEN_1_NODES=128
HIDDEN_2_NODES=64 
EMBEDDING_DIMENSION=32

I've made no effort to optimize the number of layers or the number of nodes per layer. I've also made no effort to optimize the nonlinear squashing function.

---
---

#### Graph Autoencoder

**Architecture:** Define a GAE model with two hidden layers and relu's between the hidden layers.

In [ ]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden1, hidden2, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden1)
        self.conv2 = GCNConv(hidden1, hidden2)
        self.conv3 = GCNConv(hidden2, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        return x

Let's discuss the elements of this class, both as a review and because each GNN we define next uses the same architecture pattern.

The `__init__` function inherits from the `torch.nn.Module` superclass and instantiates three convolutional layers: one for the first hidden layer, one for the second hidden layer, and one for the output layer.

The `forward` function chains the outputs from one layer into the inputs of the next. Stated simply, it passes the input forward through each layer, applying nonlinearities between layers.

**Training:** Add the training function, which just sequences the steps required for training. They key part of the training function is the definition of loss: `loss = model.recon_loss(z, data.edge_index, negative_samples)`. This says to use the reconstruction loss. In case you've forgotten, this just means that the goal of this network is to take the embedding and turn it into the the adjacency matrix using the following steps. Recall that we need to include negative samples (pairs of vertices not connected by an edge) to get good performance.

Recall that the goal is to take node $i$, call it $u_i$, and compute a real-valued vector representation of the node, call it ${\mathbf z}_i$. We want the embedding to satisfy the property that two similar nodes end up close to each other in the embedding space.

- if ${\rm sim}(u_i, u_j)$ is high then ${\mathbf z}_i$ is near ${\mathbf z}_j$.

For a graph convolutional autoencoder, we need to define what we mean both by _similar_ and by _near_. 

- We'll use adjacency to define _similar_, so two nodes are similar if $A_{ij}=1$.
- We'll use cosine similarity as the metric for _near_, so we want ${\mathbf z}_i^T{\mathbf z}_j$ to be high.

The maximum value of the cosine between two vectors is 1, and the minimum value for the cosine between two vectors is -1. And since we aren't dividing by the length of ${\mathbf z}_i$ and ${\mathbf z}_j$ like we technicall have to do if we want the vector product to represent actual cosine, we aren't guaranteed that ${\mathbf z}_i^T{\mathbf z}_j$ will even be between $-1$ and $1$. To fix this, we'll pass this product through the sigmoid function, which means that _near_ is defined as

$$ \sigma({\mathbf z}_i^T{\mathbf z}_j) = \frac{1}{1+e^{-{\mathbf z}_i^T{\mathbf z}_j}}$$

which squashes the values of _near_ so that they are always between $0$ and $1$. In other words, we'll approximate $A_{ij}$ by $\sigma({\mathbf z}_i^T{\mathbf z}_j)$.  The decoder does this computation for us.

In [ ]:
from torch import Tensor

def train(model: GAE,
          data: PyGData, 
          negative_samples: Tensor,
          optimizer: torch.optim.Optimizer
        ) -> float:
          
    model.train()
    optimizer.zero_grad()
    z = model.encode(data.x, data.edge_index)
    loss = model.recon_loss(z, data.edge_index, negative_samples)
    loss.backward()
    optimizer.step()
    return loss.item()

Set up the model and train. 
- The encoder instantiates the GCN architecture using the required number of input and hidden layers. Its goal is to output the embedding.
- The `model` is set to `GAE(encoder)`. This is the GAE decoder, which tries to produce the adjacency matrix using the math described above
- The `model.recon_loss` uses a specific loss function designed to efficiently compute error between the actual adjacency matrix and the predicted adjacency matrix.

In [ ]:
from torch_geometric.utils import negative_sampling
# Get some negative samples
neg_edge_index = negative_sampling(
    edge_index=data_onehot.edge_index,       # existing edges
    num_nodes=data_onehot.num_nodes,         # number of nodes
    num_neg_samples=data_onehot.edge_index.size(1)  # match # of positive samples
)

# Parameters
num_epochs: int = 600

# Model setup
encoder = GCNEncoder(in_channels=data_onehot.num_node_features, hidden1=HIDDEN_1_NODES, hidden2=HIDDEN_2_NODES, out_channels=EMBEDDING_DIMENSION)
model = GAE(encoder)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(1, num_epochs+1):
    loss = train(model, data_onehot, neg_edge_index, optimizer)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f}")

The embedding dimension is pretty big (32 dimensions) so we can't visualize it very well. We can get a sense for whether or not there is good clustering by projecting the 32 dimensions down to 2 dimensions using the TSNE algorithm, which tries to choose the values in the 2 dimensions so that the result is easy to visualize. The plot therefore gives us a sense of whether the embedding can be used to identify similar nodes by how well they cluster in the embedding space. 

In [ ]:
# Put the mode in evaluation (instead of training) mode
model.eval()

# Tell PyTorch to not track gradients and then infer the embedding
with torch.no_grad():
    node_embeddings_GAE = model.encode(data_onehot.x, data_onehot.edge_index)  # shape: [num_nodes, embedding_dim]

plot_scatter(node_embeddings_GAE, data_onehot.y.cpu(), title="t-SNE 2D Projection of GAE Node Embeddings")

## Question 1

Answer this question before running the next cell.

Based on patterns you see in the scatterplot above, what will happen when the 32-dimensional embeddings are clustered into 6 clusters (since the seventh class never appears in the data)? In other words, suppose that we find six clusters and then label each node by its cluster. How well will these node labels correspond to the actual node labels?

Use this mini scientific template in your response:
1. **Hypothesis**: your expected outcome.
2. **Mechanism**: why this should happen in terms of graph aggregation/message passing.
3. **Observable prediction**: what pattern you expect in the side-by-side graph comparison.
4. **Falsifier**: what result would show your hypothesis is wrong.

Also cite one concept from the GCN overview tutorial (for example, local aggregation scope or message type).

## Answer 1

Put your answer here

---

## Question 2

Run the following cell, which shows a side-by-side comparison of the node classes of ground truth versus the labels produced by the GAE. What did you get right in your answer to Problem 1? What did you get wrong?

In [ ]:
# Run KMeans on the learned embeddings node_embeddings
kmeans = KMeans(n_clusters=6, init="random", n_init=10, random_state=1234)

# Assign nodes to classes according to which cluster they belong
cluster_labels_GAE = kmeans.fit_predict(node_embeddings_GAE.cpu().numpy())  # Assuming z is a torch tensor

# Get true labels
true_labels = data_onehot.y.cpu().numpy()  # Ground truth labels from PyG Data object

# Plot side-by-side comparison
_, axes = plt.subplots(1, 2, figsize=(14, 6))

# Show both cluster and true plots side by side
plot_graph(G_sub, cluster_labels_GAE, title = "KMeans Cluster Labels from GAE Embeddings", ax=axes[1])
plot_graph(G_sub, true_labels, title = "True Classes", ax=axes[0])


## Answer 2

Put your answer to question 2 here

---
---

#### Using Node Features in GAE


Set up a graph convolutional neural network that uses the feature vectors but no other training data. We'll use the same model just a different set of input data. Thus, there is no need to define the encoder and training function again. We'll instantiate the model, train, and look at the results.

In [ ]:
# Model setup
encoder = GCNEncoder(in_channels=data_cora.num_node_features, hidden1=HIDDEN_1_NODES, hidden2=HIDDEN_2_NODES, out_channels=EMBEDDING_DIMENSION)
model = GAE(encoder)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(1, num_epochs+1):
    loss = train(model, data_cora, neg_edge_index, optimizer)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    node_embeddings_GAE_features = model.encode(data_cora.x, data_cora.edge_index)


Visualize whether the result is likely to cluster

In [ ]:
plot_scatter(node_embeddings_GAE_features, 
             data_cora.y.cpu(),
             title="t-SNE 2D Projection of GAE Embeddings with Feature Vector")

## Question 3

Answer this question before running the next cell.

Based on patterns you see in the scatterplot above, what will happen when the 32-dimensional embeddings are clustered into 6 clusters (since the seventh class never appears in the data)? In other words, suppose that we find six clusters and then label each node by its cluster. How well will these node labels correspond to the actual node labels?

Use this mini scientific template in your response:
1. **Hypothesis**
2. **Mechanism**: explicitly discuss how reconstruction loss in a graph autoencoder influences the embedding.
3. **Observable prediction**
4. **Falsifier**

Also cite one concept from the graph autoencoder tutorial (for example, encoding/decoding or reconstruction objective).

## Answer 3

Put your answer here

---

## Question 4

Run the following cell, which shows a side-by-side comparison of the node classes of ground truth versus the labels produced by the GAE. What did you get right in your answer to Problem 3? What did you get wrong?

In [ ]:
# Run KMeans on the learned embeddings node_embeddings
kmeans = KMeans(n_clusters=6, init="random", n_init=10, random_state=1234)

# Assign nodes to classes according to which cluster they belong
cluster_labels_gae_cora = kmeans.fit_predict(node_embeddings_GAE_features.cpu().numpy())  # Assuming z is a torch tensor

# Get true labels
true_labels = data_cora.y.cpu().numpy()  # Ground truth labels from PyG Data object

# Plot side-by-side comparison
_, axes = plt.subplots(1, 2, figsize=(14, 6))

# Show both cluster and true plots side by side
plot_graph(G_sub, cluster_labels_gae_cora, title = "KMeans Cluster Labels from GAE Embeddings on Cora features", ax=axes[1])
plot_graph(G_sub, true_labels, title = "True Classes", ax=axes[0])


## Answer 4

Put your answer here

### Control Condition: Chance-Level Baseline

Before moving into semi-supervised classification, run a quick control experiment.

#### ARI: Quantifying Performance

The Adjusted Rand Index (ARI) measures how similar two groupings are, adjusting for random chance. Prompting ChatGPT-4o notes that ARI evaluates

- Which pairs of points were grouped together
- Which pairs were placed in different groups
- Whether this matches what happened in the ground truth labels

ChatGPT also suggests the following interpretations for ARI ranges.

**Adjusted Rand Index (ARI) Interpretation**

| ARI Score | Interpretation                       |
|-----------|--------------------------------------|
| **1.0**   | Perfect match with true labels       |
| 0.8–0.9   | Excellent clustering                  |
| 0.5–0.7   | Good structure, moderate alignment    |
| 0.2–0.4   | Weak clustering, some structure       |
| **0.0**   | No better than random assignment      |
| **< 0.0** | Worse than random (actively misleading) |


### Question 5: Control Question

A useful sanity check is to compare against a deliberately uninformative baseline: shuffled labels. If your ARI against shuffled labels is near 0, that is evidence the ARI metric is behaving as expected.

Write a one-sentence prediction before running the next cell: what ARI do you expect for shuffled labels, and why?

Answer 5: 


---

In [ ]:
# Control condition: compare true labels to shuffled labels (chance baseline)
true_labels = data_cora.y.cpu().numpy()
rng = np.random.default_rng(123)
shuffled_labels = rng.permutation(true_labels)

ari_shuffled_control = adjusted_rand_score(true_labels, shuffled_labels)
print(f"Control ARI (true labels vs shuffled labels): {ari_shuffled_control:.4f}")

---
---

#### Adding semi-supervised learning to the GCN


Set up the GCN to do learning, but with classifier error. We have to define a new object for two reasons:
- the dimension of the output layer won't be the embedding dimension but rather the number of desired node classes.
- we want to be able to output the hidden layer since that layer is the node embedding

To make the network as similar to the previous networks as possible, I'll add another hidden layer that will have the same dimension as the previous models.

In [ ]:
# Define GCN with classifier output
class GCNClassifier(torch.nn.Module):
    def __init__(self, in_channels, hidden1, hidden2, hidden3, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden1)
        self.conv2 = GCNConv(hidden1, hidden2)
        self.conv3 = GCNConv(hidden2, hidden3)
        self.conv4 = GCNConv(hidden3, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        hidden = F.relu(x)
        x = self.conv4(hidden, edge_index)
        return x, hidden


Notice the last three lines of the `forward` method. We explicitly identify the matrix in the `hidden` layer and return that along with the output of the last layer. The output of the last layer has the dimension of the number of classes and is used to estimate class.

The training will use a different error since the goal is not to find the 32-dimensional embedding but rather to estimate the node's class. 

In [ ]:
def train_epoch(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out, _ = model(data)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        out, hidden = model(data)
        pred = out.argmax(dim=1)

        train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
        test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

    return out, hidden, train_acc, val_acc, test_acc


The key difference is that we use labeled training nodes and compute cross-entropy loss, which measures the difference between predicted classes and true classes. We then monitor train, validation, and test performance using their respective masks.

We can now instantiate the class and train it. Recall how the autoencode instantiated a model: 

`encoder = GCNEncoder(in_channels=data_onehot.num_node_features, hidden1=HIDDEN_1_NODES, hidden2=HIDDEN_2_NODES, out_channels=EMBEDDING_DIMENSION)`
sets the number of output channels to the embedding dimension followed by the instantiation 

`model = GAE(encoder)`

which creates the model as a graph autoencoder.

The classifier model uses `out_channels=num_classes` and instantiates the entire model.

In [ ]:
# Instantiate model
model = GCNClassifier(
    in_channels=data_cora.num_node_features,
    hidden1=HIDDEN_1_NODES,
    hidden2=HIDDEN_2_NODES,
    hidden3=EMBEDDING_DIMENSION,
    out_channels=num_classes
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

best_val_acc = -1.0
best_state = None

for epoch in range(1, 201):
    loss = train_epoch(model, data_cora, optimizer)
    out, hidden, train_acc, val_acc, test_acc = evaluate(model, data_cora)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 20 == 0:
        print(
            f"Epoch {epoch:3d} | Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}"
        )

# Restore best validation model before final analysis.
if best_state is not None:
    model.load_state_dict(best_state)


We output accuracy on the training, validation, and test masks. The line `pred = out.argmax(dim=1)` chooses the predicted class by finding the class with the highest output score for each node.

Visualize results, comparing what happens when we use the embedding plus some clusters to what happens when we just run the classifier. Notice that the code does the clustering on the output of the hidden layer instead of the output layer. 

In [ ]:
# Extract logits and embeddings from the best-validation model
model.eval()
with torch.no_grad():
    logit, node_embeddings = model(data_cora)

predicted_labels = logit.argmax(dim=1)
train_acc = (predicted_labels[data_cora.train_mask] == data_cora.y[data_cora.train_mask]).float().mean().item()
val_acc = (predicted_labels[data_cora.val_mask] == data_cora.y[data_cora.val_mask]).float().mean().item()
test_acc = (predicted_labels[data_cora.test_mask] == data_cora.y[data_cora.test_mask]).float().mean().item()

print("Embeddings shape:", node_embeddings.shape)
print("Label shape:", data_cora.y.shape)
print(f"Best-model Train/Val/Test accuracy = {train_acc:.4f} / {val_acc:.4f} / {test_acc:.4f}")

plot_scatter(
    node_embeddings,
    data_cora.y.cpu(),
    title="t-SNE 2D Projection of Classifier Embedding with Feature Vector"
)


## Question 6

Answer this question before running the next cell.

The embedding produced above uses information about known node classes through `train_mask` labels. How well will we be able to find clusters in the embedding and use those to predict the class of each node? And how well will this prediction compare to using classifier logits directly?

Use this mini scientific template in your response:
1. **Hypothesis**
2. **Mechanism**: explain in terms of semi-supervised learning with masks.
3. **Observable prediction**
4. **Falsifier**

Also cite one concept from the semi-supervised tutorial (for example, role of `train_mask`, `val_mask`, and `test_mask`).

## Answer 6

Run the following code and look at the clusters.

In [ ]:
# --- Clustering on embeddings ---
kmeans = KMeans(n_clusters=num_classes, random_state=0)
cluster_labels_classifier_embedding = kmeans.fit_predict(node_embeddings.cpu().numpy())

# --- Classification prediction (use logits, not embeddings) ---
predicted_labels = logit.argmax(dim=1).cpu().numpy()

# --- True labels ---
true_labels = data_cora.y.cpu().numpy()

_, axes = plt.subplots(1, 3, figsize=(18, 6))

plot_graph(G_sub, true_labels, title="True Labels", ax=axes[0])
plot_graph(G_sub, cluster_labels_classifier_embedding, title="KMeans Clusters on GCN Embeddings", ax=axes[1])
plot_graph(G_sub, predicted_labels, title="GCN Predicted Labels (argmax logits)", ax=axes[2])


## Question 7

What did you get right and wrong in your answer to problem 5? What did you learn?

## Answer 7

---
---

## Question 8

Answer this question before running the code.

Put the following four ways of classifying nodes in order from smallest to largest ARI scores. Justify your answer.
- Classes from GAE with one-hot encoding
- Classes from GAE with bag-of-words node feature encoding
- Classes from GCN classifier with clustered embedding
- Classes from GCN classifier output logits

In your justification, explicitly connect your ranking to all three tutorial ideas:
- aggregation/message passing (GCN overview)
- reconstruction objective (GAE tutorial)
- semi-supervised masks and classifier supervision (semi-supervised tutorial)

## Answer 8

Put your answer here.

---

Compute and print out the ARI for each classifier.

Compute the ARI for the GAE

In [ ]:
# ARI for one representative run
ari_gae_onehot = adjusted_rand_score(data_onehot.y.cpu().numpy(), cluster_labels_GAE)
ari_gae_features = adjusted_rand_score(data_onehot.y.cpu().numpy(), cluster_labels_gae_cora)
ari_kmeans_classifier = adjusted_rand_score(true_labels, cluster_labels_classifier_embedding)
ari_classifier = adjusted_rand_score(true_labels, predicted_labels)

print(f"ARI (GAE + one-hot): {ari_gae_onehot:.4f}")
print(f"ARI (GAE + bag-of-words): {ari_gae_features:.4f}")
print(f"ARI (KMeans on GCN embeddings): {ari_kmeans_classifier:.4f}")
print(f"ARI (GCN classifier logits): {ari_classifier:.4f}")

# Control condition: shuffled labels should produce ARI near 0.
rng = np.random.default_rng(123)
shuffled_labels = rng.permutation(true_labels)
ari_control = adjusted_rand_score(true_labels, shuffled_labels)
print(f"Control ARI (true labels vs shuffled labels): {ari_control:.4f}")

# Stability check over multiple seeds for methods that use KMeans.
kmeans_seeds = [0, 1, 2, 3, 4]
ari_gae_onehot_runs = []
ari_gae_features_runs = []
ari_kmeans_classifier_runs = []

for seed in kmeans_seeds:
    c1 = KMeans(n_clusters=6, init="random", n_init=10, random_state=seed).fit_predict(node_embeddings_GAE.cpu().numpy())
    c2 = KMeans(n_clusters=6, init="random", n_init=10, random_state=seed).fit_predict(node_embeddings_GAE_features.cpu().numpy())
    c3 = KMeans(n_clusters=num_classes, init="random", n_init=10, random_state=seed).fit_predict(node_embeddings.cpu().numpy())

    ari_gae_onehot_runs.append(adjusted_rand_score(data_onehot.y.cpu().numpy(), c1))
    ari_gae_features_runs.append(adjusted_rand_score(data_onehot.y.cpu().numpy(), c2))
    ari_kmeans_classifier_runs.append(adjusted_rand_score(true_labels, c3))

print("\nMean ± std over KMeans seeds")
print(f"GAE + one-hot: {np.mean(ari_gae_onehot_runs):.4f} ± {np.std(ari_gae_onehot_runs):.4f}")
print(f"GAE + bag-of-words: {np.mean(ari_gae_features_runs):.4f} ± {np.std(ari_gae_features_runs):.4f}")
print(f"KMeans on GCN embeddings: {np.mean(ari_kmeans_classifier_runs):.4f} ± {np.std(ari_kmeans_classifier_runs):.4f}")
print(f"GCN classifier logits: {ari_classifier:.4f} ± 0.0000 (single trained model)")


## Question 9

What did you get right and what did you get wrong? Why might your original answer have been wrong?

Then run two follow-up experiments:
1. Change `train_percent` from 10% to 30% and re-run the classifier pipeline.
2. Repeat clustering with at least 5 random seeds (already scaffolded below for KMeans) and compare mean ± std ARI.

Finally, discuss what changed and what did not change, and why.

## Answer 9

Put your answer here.

## Question 10: Final Synthesis

Given your evidence from all experiments, when would you choose each of the following in practice?
- GAE with one-hot input
- GAE with bag-of-words features
- Semi-supervised GCN classifier

For each choice, state:
1. One situation where it is a good fit.
2. One limitation or failure mode.
3. One concrete next step you would take to improve performance.

## Answer 10: 

Put your answer here